In [22]:
#imports and config

from pathlib import Path
import json
import re
import uuid
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import duckdb


In [23]:
# paths
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

BRONZE_ROOT = PROJECT_ROOT / "data/bronze/comtrade/monthly_history"
AUDIT_OUTPUT = PROJECT_ROOT / "data/metadata/comtrade/ingest_reports"
AUDIT_OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"BRONZE_ROOT: {BRONZE_ROOT}")
print(f"BRONZE_ROOT exists: {BRONZE_ROOT.exists()}")
print(f"AUDIT_OUTPUT: {AUDIT_OUTPUT}")

PROJECT_ROOT: /Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly
BRONZE_ROOT: /Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/bronze/comtrade/monthly_history
BRONZE_ROOT exists: True
AUDIT_OUTPUT: /Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/metadata/comtrade/ingest_reports


In [24]:
import pandas as pd
import json
from pathlib import Path
from typing import List, Dict, Any

def list_comtrade_json_files(base_path: Path) -> List[Path]:
    """Recursively list all JSON files under the Comtrade bronze root."""
    files = sorted(base_path.rglob("*.json"))
    print(f"Found {len(files)} JSON files")
    return files

def load_comtrade_data_to_dataframe(file_paths: List[Path]) -> pd.DataFrame:
    """Read JSON files and combine payload records into a single DataFrame."""
    all_records = []

    for idx, json_file in enumerate(file_paths, 1):
        try:
            with open(json_file, "r", encoding="utf-8") as f:
                data = json.load(f)

            if isinstance(data, dict) and "data" in data:
                records = data["data"]
                total_count = data.get("count")

                for record in records:
                    record["source_file"] = json_file.name
                    record["source_year"] = json_file.parent.name
                    record["total_count"] = total_count

                all_records.extend(records)

                if idx % 50 == 0:
                    print(f"Processed {idx}/{len(file_paths)} files, total records: {len(all_records)}")
            else:
                print(f"Unexpected structure in {json_file}")

        except Exception as e:
            print(f"Error reading {json_file}: {e}")
            continue

    print(f"\nCreating DataFrame with {len(all_records)} total records...")
    df = pd.DataFrame(all_records)
    print(f"DataFrame shape: {df.shape}")
    return df

# Usage
json_file_paths = list_comtrade_json_files(BRONZE_ROOT)
print("First 3 files:")
for p in json_file_paths[:3]:
    print(f" - {p}")

df = load_comtrade_data_to_dataframe(json_file_paths)
df.head()

Found 288 JSON files
First 3 files:
 - /Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/bronze/comtrade/monthly_history/year=2020/hist_batch_reps_100_cmd_1001-1005-1006-1201-2709-2710_periods_202001_to_202004_flow_M_20260312_121526.json
 - /Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/bronze/comtrade/monthly_history/year=2020/hist_batch_reps_100_cmd_1001-1005-1006-1201-2709-2710_periods_202001_to_202004_flow_X_20260312_121540.json
 - /Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/bronze/comtrade/monthly_history/year=2020/hist_batch_reps_100_cmd_1001-1005-1006-1201-2709-2710_periods_202005_to_202008_flow_M_20260312_121555.json
Processed 50/288 files, total records: 162414
Processed 100/288 files, total records: 339305
Processed 150/288 files, total records: 537573
Processed 200/288 files, total records: 756372
Processed 250/288 files, total records: 968968

Creating DataFrame with 108

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate,source_file,source_year,total_count
0,C,M,20200101,2020,1,202001,100,BGR,Bulgaria,M,...,False,1.058049e+06,0.0,1.058049e+06,0,False,True,hist_batch_reps_100_cmd_1001-1005-1006-1201-27...,year=2020,1398
1,C,M,20200101,2020,1,202001,100,BGR,Bulgaria,M,...,False,1.028874e+07,0.0,1.028874e+07,0,False,True,hist_batch_reps_100_cmd_1001-1005-1006-1201-27...,year=2020,1398
2,C,M,20200101,2020,1,202001,100,BGR,Bulgaria,M,...,False,1.634398e+05,0.0,1.634398e+05,0,False,True,hist_batch_reps_100_cmd_1001-1005-1006-1201-27...,year=2020,1398
3,C,M,20200101,2020,1,202001,100,BGR,Bulgaria,M,...,False,5.793973e+05,0.0,5.793973e+05,0,False,True,hist_batch_reps_100_cmd_1001-1005-1006-1201-27...,year=2020,1398
4,C,M,20200101,2020,1,202001,100,BGR,Bulgaria,M,...,False,3.848297e+04,0.0,3.848297e+04,0,False,True,hist_batch_reps_100_cmd_1001-1005-1006-1201-27...,year=2020,1398


In [25]:
#schema audit
def parse_filename_metadata(file_path: Path) -> dict:
    name = file_path.name

    year_match = re.search(r"year=(\d{4})", str(file_path))
    reporter_match = re.search(r"reps_(\d+)", name)
    flow_match = re.search(r"flow_([MX])", name)
    periods_match = re.search(r"periods_(\d{6})_to_(\d{6})", name)
    cmd_match = re.search(r"cmd_([0-9\-]+)", name)
    ts_match = re.search(r"_(\d{8}_\d{6})\.json$", name)

    return {
        "source_file": name,
        "source_path": str(file_path),
        "source_year_partition": int(year_match.group(1)) if year_match else None,
        "requested_reporter_code": reporter_match.group(1) if reporter_match else None,
        "requested_flow_code": flow_match.group(1) if flow_match else None,
        "requested_period_start": periods_match.group(1) if periods_match else None,
        "requested_period_end": periods_match.group(2) if periods_match else None,
        "requested_cmd_codes": cmd_match.group(1).split("-") if cmd_match else None,
        "filename_extract_ts": ts_match.group(1) if ts_match else None,
    }


In [26]:
#null audit
def load_comtrade_json(file_path: Path) -> dict:
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)


In [27]:
audit_rows = []

# 1) Reuse earlier extracted df only when it contains source_file metadata; otherwise rebuild
can_reuse_df = (
    "df" in globals()
    and isinstance(df, pd.DataFrame)
    and not df.empty
    and "source_file" in df.columns
)

if can_reuse_df:
    extracted_df = df.copy()
    print(f"Using existing extracted df from earlier cells: {extracted_df.shape}")
else:
    if "json_file_paths" not in globals() or not json_file_paths:
        json_file_paths = list_comtrade_json_files(BRONZE_ROOT)
    extracted_df = load_comtrade_data_to_dataframe(json_file_paths)
    print(f"Rebuilt extracted df using JSON extraction function: {extracted_df.shape}")

if "json_file_paths" not in globals() or not json_file_paths:
    json_file_paths = list_comtrade_json_files(BRONZE_ROOT)

if "source_file" not in extracted_df.columns:
    raise ValueError(
        "Expected column 'source_file' is missing in extracted_df. "
        "Check that load_comtrade_data_to_dataframe adds source_file metadata."
    )

# 2) Build per-file metadata table and attach to each row in extracted_df
file_meta_rows = []
for file_path in json_file_paths:
    obj = load_comtrade_json(file_path)
    meta = obj.get("_metadata", {})
    request = meta.get("request", {})

    meta_row = parse_filename_metadata(file_path)
    meta_row.update({
        "request_reporter_code": request.get("reporterCode"),
        "request_periods": request.get("period"),
        "request_flow_code": request.get("flowCode"),
        "request_cmd_code": request.get("cmdCode"),
        "bronze_extracted_at": meta.get("extracted_at_utc"),
    })
    file_meta_rows.append(meta_row)

file_meta_df = (
    pd.DataFrame(file_meta_rows)
    .drop_duplicates(subset=["source_file"], keep="last")
    .loc[:, [
        "source_file",
        "source_path",
        "source_year_partition",
        "requested_reporter_code",
        "requested_flow_code",
        "requested_period_start",
        "requested_period_end",
        "requested_cmd_codes",
        "filename_extract_ts",
        "request_reporter_code",
        "request_periods",
        "request_flow_code",
        "request_cmd_code",
        "bronze_extracted_at",
    ]]
)

df_for_audit = extracted_df.merge(file_meta_df, on="source_file", how="left")
print(f"df_for_audit shape (row-level + metadata): {df_for_audit.shape}")

# 3) File-level audit, computed from metadata-enriched rows
for file_path in json_file_paths:
    obj = load_comtrade_json(file_path)
    data = obj.get("data", [])
    meta = obj.get("_metadata", {})
    request = meta.get("request", {})

    file_df = df_for_audit[df_for_audit["source_file"] == file_path.name].copy()
   

    row = parse_filename_metadata(file_path)
    row.update({
        "elapsed_time": obj.get("elapsedTime"),
        "count_from_json": obj.get("count"),
        "record_count_metadata": meta.get("record_count"),
        "bronze_extracted_at": meta.get("extracted_at_utc"),
        "request_reporter_code": request.get("reporterCode"),
        "request_periods": request.get("period"),
        "request_flow_code": request.get("flowCode"),
        "request_cmd_code": request.get("cmdCode"),
        "row_count": len(file_df),
        "min_period_present": file_df["period"].min() if "period" in file_df.columns and not file_df.empty else None,
        "max_period_present": file_df["period"].max() if "period" in file_df.columns and not file_df.empty else None,
        "classification_codes_present": sorted(file_df["classificationCode"].dropna().unique().tolist()) if "classificationCode" in file_df.columns and not file_df.empty else [],
        "cmd_codes_present": sorted(file_df["cmdCode"].astype(str).dropna().unique().tolist()) if "cmdCode" in file_df.columns and not file_df.empty else [],
        "reporters_present": sorted(file_df["reporterISO"].dropna().unique().tolist()) if "reporterISO" in file_df.columns and not file_df.empty else [],
        "flows_present": sorted(file_df["flowCode"].dropna().unique().tolist()) if "flowCode" in file_df.columns and not file_df.empty else [],
    })

    # Null rates for key fields
    for col in ["qty", "altQty", "netWgt", "grossWgt", "cifvalue", "fobvalue", "primaryValue"]:
        row[f"null_rate__{col}"] = file_df[col].isna().mean() if col in file_df.columns and not file_df.empty else None

    # Duplicate rate at expected raw grain
    if not file_df.empty:
        dup_cols = ["period", "reporterISO", "partnerISO", "flowCode", "cmdCode", "classificationCode", "customsCode","motCode","partner2Code"]
        existing_dup_cols = [c for c in dup_cols if c in file_df.columns]
        if existing_dup_cols:
            duplicate_count = file_df.duplicated(subset=existing_dup_cols).sum()
            row["duplicate_count"] = int(duplicate_count)
            row["duplicate_rate"] = duplicate_count / len(file_df)
        else:
            row["duplicate_count"] = None
            row["duplicate_rate"] = None
    else:
        row["duplicate_count"] = None
        row["duplicate_rate"] = None

    audit_rows.append(row)

audit_df = pd.DataFrame(audit_rows)
print(audit_df.shape)
audit_df.head()

Using existing extracted df from earlier cells: (1085896, 50)
df_for_audit shape (row-level + metadata): (1085896, 63)
(288, 33)


,source_file,source_path,source_year_partition,requested_reporter_code,requested_flow_code,requested_period_start,requested_period_end,requested_cmd_codes,filename_extract_ts,elapsed_time,...,flows_present,null_rate__qty,null_rate__altQty,null_rate__netWgt,null_rate__grossWgt,null_rate__cifvalue,null_rate__fobvalue,null_rate__primaryValue,duplicate_count,duplicate_rate
0,hist_batch_reps_100_cmd_1001-1005-1006-1201-27...,/Users/chromazone/Documents/Python/Data Engine...,2020,100,M,202001,202004,"[1001, 1005, 1006, 1201, 2709, 2710]",20260312_121526,5.86 secs,...,[M],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,hist_batch_reps_100_cmd_1001-1005-1006-1201-27...,/Users/chromazone/Documents/Python/Data Engine...,2020,100,X,202001,202004,"[1001, 1005, 1006, 1201, 2709, 2710]",20260312_121540,2.49 secs,...,[X],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,hist_batch_reps_100_cmd_1001-1005-1006-1201-27...,/Users/chromazone/Documents/Python/Data Engine...,2020,100,M,202005,202008,"[1001, 1005, 1006, 1201, 2709, 2710]",20260312_121555,3.34 secs,...,[M],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,hist_batch_reps_100_cmd_1001-1005-1006-1201-27...,/Users/chromazone/Documents/Python/Data Engine...,2020,100,X,202005,202008,"[1001, 1005, 1006, 1201, 2709, 2710]",20260312_121609,2.38 secs,...,[X],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,hist_batch_reps_100_cmd_1001-1005-1006-1201-27...,/Users/chromazone/Documents/Python/Data Engine...,2020,100,M,202009,202012,"[1001, 1005, 1006, 1201, 2709, 2710]",20260312_121629,7.97 secs,...,[M],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [28]:
audit_df.to_csv(AUDIT_OUTPUT / "comtrade_bronze_audit.csv", index=False)
audit_df.to_parquet(AUDIT_OUTPUT / "comtrade_bronze_audit.parquet", index=False)


In [29]:
audit_df.groupby("source_year_partition")["row_count"].sum()


source_year_partition
2020    159186
2021    172179
2022    194212
2023    217518
2024    216697
2025    126104
Name: row_count, dtype: int64

In [31]:
audit_df.explode("classification_codes_present").groupby("classification_codes_present")["row_count"].sum()


classification_codes_present
H5    331365
H6    754531
Name: row_count, dtype: int64

In [32]:
audit_df[[
    "source_file",
    "row_count",
    "duplicate_rate",
    "null_rate__netWgt",
    "null_rate__cifvalue",
    "null_rate__fobvalue"
]].sort_values("duplicate_rate", ascending=False).head(20)


,source_file,row_count,duplicate_rate,null_rate__netWgt,null_rate__cifvalue,null_rate__fobvalue
0,hist_batch_reps_100_cmd_1001-1005-1006-1201-27...,1398,0.0,0.000000,0.0,0.0
182,hist_batch_reps_842_cmd_1001-1005-1006-1201-27...,594,0.0,0.000000,0.0,0.0
188,hist_batch_reps_97_cmd_1001-1005-1006-1201-270...,2765,0.0,0.002893,0.0,1.0
187,hist_batch_reps_97_cmd_1001-1005-1006-1201-270...,3912,0.0,0.004090,1.0,0.0
186,hist_batch_reps_97_cmd_1001-1005-1006-1201-270...,2760,0.0,0.002899,0.0,1.0
185,hist_batch_reps_842_cmd_1001-1005-1006-1201-27...,1554,0.0,0.000000,1.0,0.0
184,hist_batch_reps_842_cmd_1001-1005-1006-1201-27...,630,0.0,0.003175,0.0,0.0
183,hist_batch_reps_842_cmd_1001-1005-1006-1201-27...,1496,0.0,0.000000,1.0,0.0
181,hist_batch_reps_842_cmd_1001-1005-1006-1201-27...,1568,0.0,0.000000,1.0,0.0
173,hist_batch_reps_642_cmd_1001-1005-1006-1201-27...,5486,0.0,0.002917,1.0,0.0


In [33]:
print("cwd:", Path.cwd())
print("BRONZE_ROOT:", BRONZE_ROOT)
print("BRONZE_ROOT exists:", BRONZE_ROOT.exists())
print("json_file_paths count:", len(json_file_paths) if "json_file_paths" in globals() else "missing")
print("audit_df shape:", audit_df.shape if "audit_df" in globals() else "missing")
print("audit_df columns:", list(audit_df.columns) if "audit_df" in globals() else "missing")

cwd: /Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/notebooks/comtrade
BRONZE_ROOT: /Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/bronze/comtrade/monthly_history
BRONZE_ROOT exists: True
json_file_paths count: 288
audit_df shape: (288, 33)
audit_df columns: ['source_file', 'source_path', 'source_year_partition', 'requested_reporter_code', 'requested_flow_code', 'requested_period_start', 'requested_period_end', 'requested_cmd_codes', 'filename_extract_ts', 'elapsed_time', 'count_from_json', 'record_count_metadata', 'bronze_extracted_at', 'request_reporter_code', 'request_periods', 'request_flow_code', 'request_cmd_code', 'row_count', 'min_period_present', 'max_period_present', 'classification_codes_present', 'cmd_codes_present', 'reporters_present', 'flows_present', 'null_rate__qty', 'null_rate__altQty', 'null_rate__netWgt', 'null_rate__grossWgt', 'null_rate__cifvalue', 'null_rate__fobvalue', 'null_rate__primaryValue', 'd

In [85]:

df.to_csv("/Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/processed/comtrade_monthly_bronze_130326.csv", index=False)


## Collision Summary

In [34]:
df.shape

(1085896, 50)

In [35]:


# -----------------------------------------------------------------------------------
# CONFIG: set your intended silver grain here
# Adjust this list to match your chosen analytical grain
# -----------------------------------------------------------------------------------
grain_cols = [
    "period",
    "reporterISO",
    "partnerISO",
    "flowCode",
    "cmdCode",
    "classificationCode",
    "customsCode",
    "motCode",
    "partner2Code"
]

# Optional: if period is your canonical month key, you can use "period" instead of refYear/refMonth
# grain_cols = ["reporterCode", "partnerCode", "period", "flowCode", "cmdCode"]

# -----------------------------------------------------------------------------------
# BASIC VALIDATION
# -----------------------------------------------------------------------------------
missing_grain_cols = [c for c in grain_cols if c not in df.columns]
if missing_grain_cols:
    raise ValueError(f"These grain columns are missing from df: {missing_grain_cols}")

# -----------------------------------------------------------------------------------
# FIND DUPLICATE GRAIN GROUPS
# -----------------------------------------------------------------------------------
dup_mask = df.duplicated(subset=grain_cols, keep=False)
df_dups = df.loc[dup_mask].copy()

print(f"Total rows in df: {len(df):,}")
print(f"Rows involved in duplicate grain collisions: {len(df_dups):,}")
print(f"Duplicate groups: {df_dups.groupby(grain_cols).ngroups:,}")

# -----------------------------------------------------------------------------------
# IDENTIFY WHICH NON-GRAIN COLUMNS DIFFER INSIDE EACH DUPLICATE GROUP
# -----------------------------------------------------------------------------------
non_grain_cols = [c for c in df.columns if c not in grain_cols]

def differing_columns(group: pd.DataFrame) -> list:
    varying = []
    for col in non_grain_cols:
        # nunique(dropna=False) counts NaN as a distinct value, which is useful for QA
        if group[col].nunique(dropna=False) > 1:
            varying.append(col)
    return varying

if df_dups.empty:
    collision_summary = pd.DataFrame(
        columns=grain_cols + [
            "rows_in_group",
            "differing_columns",
            "differing_column_count",
            "differing_columns_str",
        ]
    )
else:
    collision_summary = (
        df_dups
        .groupby(grain_cols, dropna=False)
        .apply(lambda g: pd.Series({
            "rows_in_group": len(g),
            "differing_columns": (cols := differing_columns(g)),
            "differing_column_count": len(cols),
        }))
        .reset_index()
    )
    collision_summary["differing_columns_str"] = collision_summary["differing_columns"].apply(
        lambda x: ", ".join(x) if x else ""
    )

# -----------------------------------------------------------------------------------
# FULL DUPLICATE ROW OUTPUT, ORDERED BY GRAIN
# -----------------------------------------------------------------------------------
# Join summary back to the raw duplicate rows so you can inspect everything
df_dups_full = df_dups.merge(
    collision_summary[grain_cols + ["rows_in_group", "differing_columns_str"]],
    on=grain_cols,
    how="left"
)

sort_cols = grain_cols.copy()
# Add a couple of common tie-breakers if present
for extra_col in ["partner2Code", "customsCode", "motCode", "qtyUnitCode", "primaryValue"]:
    if extra_col in df_dups_full.columns and extra_col not in sort_cols:
        sort_cols.append(extra_col)

df_dups_full = df_dups_full.sort_values(sort_cols).reset_index(drop=True)

# -----------------------------------------------------------------------------------
# DISPLAY
# -----------------------------------------------------------------------------------
print("\n=== Collision summary ===")
display(
    collision_summary
    .sort_values(grain_cols)
    .reset_index(drop=True)
)

print("\n=== Full duplicate rows ordered by grain ===")
display(df_dups_full)

# -----------------------------------------------------------------------------------
# OUTPUT
# -----------------------------------------------------------------------------------
collision_summary.to_csv(AUDIT_OUTPUT / "comtrade_duplicate_collisions_summary2.csv", index=False)
df_dups_full.to_csv(AUDIT_OUTPUT / "comtrade_duplicate_collisions_full2.csv", index=False)

Total rows in df: 1,085,896
Rows involved in duplicate grain collisions: 0
Duplicate groups: 0

=== Collision summary ===


,period,reporterISO,partnerISO,flowCode,cmdCode,classificationCode,customsCode,motCode,partner2Code,rows_in_group,differing_columns,differing_column_count,differing_columns_str



=== Full duplicate rows ordered by grain ===


,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate,source_file,source_year,total_count,rows_in_group,differing_columns_str


## Freq column collision count

In [36]:
from collections import Counter

col_counter = Counter()

for cols in collision_summary["differing_columns"]:
    col_counter.update(cols)

collision_column_rank = (
    pd.DataFrame(
        [{"column": k, "duplicate_groups_where_column_varies": v} for k, v in col_counter.items()]
    )
    .sort_values("duplicate_groups_where_column_varies", ascending=False)
    .reset_index(drop=True)
)

display(collision_column_rank)

collision_column_rank.to_csv(AUDIT_OUTPUT / "comtrade_duplicate_collision_columns_rank.csv", index=False)

KeyError: 'duplicate_groups_where_column_varies'

## Grain checklist rubric

In [37]:
# -----------------------------------------------------------------------------------
# CONFIG: grain checklist level
# -----------------------------------------------------------------------------------
check_grain = ["reporterCode", "partnerCode", "refYear", "refMonth", "flowCode", "customsCode"]

missing_check_cols = [c for c in check_grain if c not in df.columns]
if missing_check_cols:
    raise ValueError(f"These checklist grain columns are missing from df: {missing_check_cols}")

# -----------------------------------------------------------------------------------
# Helper for null-rate calculation
# -----------------------------------------------------------------------------------
def null_rate(series):
    return series.isna().mean()

# -----------------------------------------------------------------------------------
# Build the rubric
# -----------------------------------------------------------------------------------
agg_dict = {
    "row_count": ("flowCode", "size"),  # any present column works here
    "distinct_cmd_codes": ("cmdCode", "nunique") if "cmdCode" in df.columns else ("flowCode", "size"),
    "duplicate_rows_on_full_grain": (
        grain_cols[0],
        lambda s: 0  # placeholder, filled after aggregation
    ),
}

# Optional quality columns to inspect if present
quality_cols = [
    "qty", "altQty", "netWgt", "grossWgt",
    "cifvalue", "fobvalue", "primaryValue"
]

available_quality_cols = [c for c in quality_cols if c in df.columns]

grouped = df.groupby(check_grain, dropna=False)

rubric = grouped.size().reset_index(name="row_count")

if "cmdCode" in df.columns:
    cmd_counts = grouped["cmdCode"].nunique(dropna=True).reset_index(name="distinct_cmd_codes")
    rubric = rubric.merge(cmd_counts, on=check_grain, how="left")

# Null rates per grain slice
for col in available_quality_cols:
    tmp = grouped[col].apply(null_rate).reset_index(name=f"null_rate__{col}")
    rubric = rubric.merge(tmp, on=check_grain, how="left")

# Duplicate count on your intended full analytical grain
dup_full = (
    df.assign(is_dup=df.duplicated(subset=grain_cols, keep=False).astype(int))
      .groupby(check_grain, dropna=False)["is_dup"]
      .sum()
      .reset_index(name="duplicate_rows_on_full_grain")
)

rubric = rubric.merge(dup_full, on=check_grain, how="left")

# -----------------------------------------------------------------------------------
# Boolean checklist flags
# -----------------------------------------------------------------------------------
rubric["has_rows"] = rubric["row_count"] > 0
rubric["has_cmd_coverage"] = rubric["distinct_cmd_codes"].fillna(0) > 0 if "distinct_cmd_codes" in rubric.columns else True
rubric["has_duplicate_collision"] = rubric["duplicate_rows_on_full_grain"].fillna(0) > 0

for col in available_quality_cols:
    rubric[f"all_null__{col}"] = rubric[f"null_rate__{col}"] == 1.0
    rubric[f"partially_null__{col}"] = rubric[f"null_rate__{col}"].between(0.0, 1.0, inclusive="neither")

# Simple overall QA status
def qa_status(row):
    if not row["has_rows"]:
        return "MISSING_GRAIN"
    if row["has_duplicate_collision"]:
        return "DUPLICATE_COLLISION"
    severe_nulls = [c for c in available_quality_cols if row.get(f"all_null__{c}", False)]
    if severe_nulls:
        return "CHECK_NULLS"
    return "OK"

rubric["qa_status"] = rubric.apply(qa_status, axis=1)

# Order for inspection
rubric = rubric.sort_values(check_grain).reset_index(drop=True)

display(rubric)

rubric.to_csv(AUDIT_OUTPUT / "comtrade_silver_grain_checklist.csv", index=False)

,reporterCode,partnerCode,refYear,refMonth,flowCode,customsCode,row_count,distinct_cmd_codes,null_rate__qty,null_rate__altQty,...,partially_null__netWgt,all_null__grossWgt,partially_null__grossWgt,all_null__cifvalue,partially_null__cifvalue,all_null__fobvalue,partially_null__fobvalue,all_null__primaryValue,partially_null__primaryValue,qa_status
0,97,0,2020,1,M,C00,6,6,0.0,0.0,...,False,False,False,False,False,True,False,False,False,CHECK_NULLS
1,97,0,2020,1,M,C01,6,6,0.0,0.0,...,False,False,False,False,False,True,False,False,False,CHECK_NULLS
2,97,0,2020,1,M,C06,3,3,0.0,0.0,...,False,False,False,False,False,True,False,False,False,CHECK_NULLS
3,97,0,2020,1,M,C20,3,3,0.0,0.0,...,False,False,False,False,False,True,False,False,False,CHECK_NULLS
4,97,0,2020,1,X,C00,6,6,0.0,0.0,...,True,False,False,True,False,False,False,False,False,CHECK_NULLS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161253,842,894,2025,5,X,C00,1,1,0.0,0.0,...,False,False,False,True,False,False,False,False,False,CHECK_NULLS
161254,842,894,2025,7,X,C00,1,1,0.0,0.0,...,False,False,False,True,False,False,False,False,False,CHECK_NULLS
161255,842,894,2025,8,X,C00,1,1,0.0,0.0,...,False,False,False,True,False,False,False,False,False,CHECK_NULLS
161256,842,894,2025,9,X,C00,1,1,0.0,0.0,...,False,False,False,True,False,False,False,False,False,CHECK_NULLS


## missing grain combinations that do not exist at all

The previous rubric only summarises what exists in the data.
If you want to explicitly see what is missing, build the full cartesian grid of expected combinations.

This is excellent for quick QA.

In [38]:
# -----------------------------------------------------------------------------------
# Build expected grid from observed dimensions
# You can replace these with your own expected lists if needed
# -----------------------------------------------------------------------------------
expected_reporters = sorted(df["reporterCode"].dropna().unique()) if "reporterCode" in df.columns else []
expected_partners  = sorted(df["partnerCode"].dropna().unique()) if "partnerCode" in df.columns else []
expected_years     = sorted(df["refYear"].dropna().unique()) if "refYear" in df.columns else []
expected_months    = sorted(df["refMonth"].dropna().unique()) if "refMonth" in df.columns else []
expected_flows     = sorted(df["flowCode"].dropna().unique()) if "flowCode" in df.columns else []

expected_grid = pd.MultiIndex.from_product(
    [expected_reporters, expected_partners, expected_years, expected_months, expected_flows],
    names=check_grain
).to_frame(index=False)

rubric_full = expected_grid.merge(
    rubric,
    on=check_grain,
    how="left"
)

rubric_full["row_count"] = rubric_full["row_count"].fillna(0)
rubric_full["has_rows"] = rubric_full["row_count"] > 0
rubric_full["qa_status"] = np.where(rubric_full["has_rows"], rubric_full["qa_status"].fillna("OK"), "MISSING_GRAIN")

display(
    rubric_full
    .sort_values(check_grain)
    .reset_index(drop=True)
)

ValueError: Length of names must match number of levels in MultiIndex.

# inspection filter 



In [39]:
# Show only the problematic grain slices
problem_rubric = rubric_full.query("qa_status != 'OK'").copy()

display(problem_rubric)

# Example: inspect one bad group in full detail
example_bad = problem_rubric.iloc[0][check_grain].to_dict()
print(example_bad)

inspect_df = df.copy()
for col, val in example_bad.items():
    inspect_df = inspect_df[inspect_df[col] == val]

inspect_df = inspect_df.sort_values(grain_cols).reset_index(drop=True)
display(inspect_df)

NameError: name 'rubric_full' is not defined